# NIDS Project - Data Exploration

**Author:** Mohammed Abdul Razzak Baig (25207938)  
**Dataset:** CICIDS2017 / UNSW-NB15

This notebook explores the preprocessed data and provides insights for the paper.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data.data_loader import load_splits, get_class_distribution
from src.visualization.plots import plot_class_distribution

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

%matplotlib inline

## 1. Load Dataset

In [ ]:
# Choose dataset
DATASET = 'cicids2017'  # or 'unsw-nb15'

# Load preprocessed splits
splits = load_splits(DATASET)

X_train = splits['X_train']
y_train = splits['y_train']
X_val = splits['X_val']
y_val = splits['y_val']
X_test = splits['X_test']
y_test = splits['y_test']

print(f"Dataset: {DATASET}")
print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

## 2. Load Class Names

In [ ]:
# Load preprocessor metadata to get class names
preprocessor_path = project_root / f'data/processed/{DATASET}_preprocessor/metadata.json'

with open(preprocessor_path) as f:
    metadata = json.load(f)

label_mapping = metadata['label_mapping']
class_names = list(label_mapping.keys())
n_classes = len(class_names)

print(f"Number of classes: {n_classes}")
print(f"\nClass names:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## 3. Class Distribution Analysis

In [ ]:
# Analyze class distribution in training set
unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)

print("Training Set Class Distribution:")
print("=" * 70)
print(f"{'Class':<30} {'Count':>12} {'Percentage':>12}")
print("=" * 70)

distribution_data = []
for cls_idx, count in zip(unique, counts):
    class_name = class_names[cls_idx]
    percentage = 100 * count / total
    print(f"{class_name:<30} {count:>12,} {percentage:>11.2f}%")
    
    distribution_data.append({
        'Class': class_name,
        'Count': count,
        'Percentage': percentage
    })

df_distribution = pd.DataFrame(distribution_data)

In [ ]:
# Visualize class distribution
plot_class_distribution(
    y_train,
    class_names=class_names,
    title=f'{DATASET.upper()} - Training Set Class Distribution'
)

## 4. Identify Rare Classes

In [ ]:
# Identify rare classes (< 5% of dataset)
RARE_THRESHOLD = 0.05

rare_classes = df_distribution[df_distribution['Percentage'] < RARE_THRESHOLD * 100]

print(f"Rare Classes (< {RARE_THRESHOLD*100}% of dataset):")
print("=" * 70)
if len(rare_classes) > 0:
    print(rare_classes.to_string(index=False))
    print(f"\n{len(rare_classes)} rare classes identified")
    print("These classes will be the focus of our evaluation!")
else:
    print("No rare classes found.")

## 5. Feature Statistics

In [ ]:
# Basic feature statistics
print(f"Number of features: {X_train.shape[1]}")
print(f"\nFeature statistics (first 10 features):")
print("=" * 70)

feature_stats = pd.DataFrame({
    'Mean': X_train[:, :10].mean(axis=0),
    'Std': X_train[:, :10].std(axis=0),
    'Min': X_train[:, :10].min(axis=0),
    'Max': X_train[:, :10].max(axis=0)
})

print(feature_stats)

## 6. Class Imbalance Ratio

In [ ]:
# Calculate imbalance ratio
max_samples = df_distribution['Count'].max()
min_samples = df_distribution['Count'].min()
imbalance_ratio = max_samples / min_samples

print(f"Class Imbalance Analysis:")
print("=" * 70)
print(f"Most common class: {df_distribution.loc[df_distribution['Count'].idxmax(), 'Class']}")
print(f"  Samples: {max_samples:,}")
print(f"\nLeast common class: {df_distribution.loc[df_distribution['Count'].idxmin(), 'Class']}")
print(f"  Samples: {min_samples:,}")
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")
print(f"\nThis severe imbalance justifies our focus on:")
print(f"  - Macro-F1 (not weighted F1 or accuracy)")
print(f"  - Per-class recall")
print(f"  - Imbalance handling techniques (SMOTE, focal loss)")

## 7. Summary for Paper

Key statistics to include in the Methodology section:

In [ ]:
summary = f"""
DATASET SUMMARY FOR PAPER:
{'='*70}

Dataset: {DATASET.upper()}

Total Samples:
  - Training: {len(y_train):,} ({len(y_train)/len(splits['y_train'])*100:.1f}%)
  - Validation: {len(y_val):,} ({len(y_val)/len(splits['y_train'])*100:.1f}%)
  - Test: {len(y_test):,} ({len(y_test)/len(splits['y_train'])*100:.1f}%)

Features: {X_train.shape[1]}
Classes: {n_classes}

Class Imbalance:
  - Imbalance ratio: {imbalance_ratio:.2f}:1
  - Rare classes (< 5%): {len(rare_classes)}

Preprocessing:
  - Dropped leaky features (IPs, ports, timestamps)
  - Handled infinite/NaN values (median imputation)
  - Removed duplicates
  - Standardized features (zero mean, unit variance)
  - Stratified train/val/test split (70/15/15)

{'='*70}
"""

print(summary)

# Save summary
output_path = project_root / f'experiments/results/{DATASET}/dataset_summary.txt'
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    f.write(summary)

print(f"\n✓ Summary saved to {output_path}")